# 密度

In [1]:
import sys, pathlib
HERE = pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path().cwd()
sys.path.insert(0, str(HERE))

import numpy as np, matplotlib.pyplot as plt
from pathlib import Path

from stateprep      import single_vortex
from spec_classic   import SpectralSolver          # 经典谱（基准）
from rk4_realspace  import RK4Solver               # ★ NEW: RK4 Solver
from qspec_pod      import QuantumPODSolver        # 改进量子，可保留
from plot_fivepanel import five_panel

# ---------- 参数 ----------
T_GRID   = np.linspace(0, np.pi/2, 5)
SIGMA    = 3.0
CENTER   = (0.0, 0.0)
OUT_DIR  = Path("figs"); OUT_DIR.mkdir(exist_ok=True)

# ---------- 初态 ----------
psi1_0, psi2_0 = single_vortex(SIGMA, CENTER)

# ---------- 求解 ----------
classic = SpectralSolver().evolve(psi1_0, psi2_0, T_GRID)
rk4     = RK4Solver()       .evolve(psi1_0, psi2_0, T_GRID)
try:
    q_plus = QuantumPODSolver().evolve(psi1_0, psi2_0, T_GRID)
except NotImplementedError:
    q_plus = rk4              # 若未实现，用 rk4 占位

# ---------- 绘图 ----------
for t in T_GRID:
    rho_c = np.abs(classic[t][0])**2 + np.abs(classic[t][1])**2
    rho_r = np.abs(rk4[t][0])   **2 + np.abs(rk4[t][1])   **2
    rho_p = np.abs(q_plus[t][0])**2 + np.abs(q_plus[t][1])**2

    # 归一化基准总质量
    rho_c *= rho_r.sum() / rho_c.sum()
    rho_p *= rho_r.sum() / rho_p.sum()

    fig = five_panel(
        rho_c,       # Classic
        rho_r,       # ↙ 现在第二列改成 RK4
        rho_p,       # Quantum+（或占位）
        labels=("Classic", "RK4", "Δ", "Quantum+", "Δ+"),
        title=f"rho comparison at t = {t:.2f}"
    )
    fig.savefig(OUT_DIR / f"compare_t{t:.2f}.png", dpi=300)
    plt.close(fig)
    print("✓ saved", f"compare_t{t:.2f}.png")


✓ saved compare_t0.00.png
✓ saved compare_t0.39.png
✓ saved compare_t0.79.png
✓ saved compare_t1.18.png
✓ saved compare_t1.57.png


# 涡度

In [ ]:

from compute_utils import compute_fluid_quantities
for t in T_GRID:
    # ------- 取三套 ψ 并算涡度 ω -------
    psi_c = classic[t]
    psi_r = rk4[t]
    psi_p = q_plus[t]

    _, _, _, vor_c = compute_fluid_quantities(*psi_c)
    _, _, _, vor_r = compute_fluid_quantities(*psi_r)
    _, _, _, vor_p = compute_fluid_quantities(*psi_p)

    # 统一尺度（可选：对齐 RMS、大值或保持原量纲）
    # 例：不做 scale，直接比较
    diff   = vor_r - vor_c
    diff_p = vor_p - vor_c

    fig = five_panel(
        vor_c,             # Classic   （连续 colormap 改成 coolwarm 也行）
        vor_r,             # RK4
        vor_p,             # Quantum+
        labels=("Classic ω", "RK4 ω", "Δ", "Quantum+ ω", "Δ+"),
        title=f"Vorticity  t = {t:.2f}"
    )
    fig.savefig(OUT_DIR / f"vorticity_t{t:.2f}.png", dpi=300)
    plt.close(fig)